# Course 2 — ML Strategy & Best Practices
=========================================

Structuring a machine learning project requires systematic data allocation, rigorous performance metrics, and diagnostic checklists to guide development path decisions.

### In this notebook, we will explore:
1. **Train / Dev / Test Splits**: Allocating datasets to guarantee unbiased evaluation.
2. **Evaluation Metrics**: Formulating Precision, Recall, F1 Score, and ROC-AUC curves.
3. **Cross-Validation**: Measuring validation stability across subsets.
4. **Error Analysis**: Structuring diagnostic lists to prioritize modeling tasks.
5. **Class Imbalance**: Evaluating model bias on highly skewed distributions and applying cost-sensitive learning.

In [ ]:
import sys
from pathlib import Path
# Add repository root to sys.path dynamically
project_root = Path(".").resolve()
while project_root.name and not (project_root / "utils").is_dir():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    roc_curve, auc
)


## 1. Train / Dev / Test Splits

To build a generalizable machine learning system, the dataset is divided into three distinct subsets:
- **Training Set**: Used to fit the parameters (weights and biases) of the models.
- **Dev (Development/Validation) Set**: Used to evaluate models, tune hyperparameters, and perform error analysis. It must reflect the target distribution.
- **Test Set**: Used purely for final, unbiased evaluation. No tuning or decision-making should be done based on test set results to avoid target leakage.

### Split Ratios:
- **Traditional (small/medium data)**: $60\% / 20\% / 20\%$ splits are common.
- **Big Data ($> 1$ million samples)**: $98\% / 1\% / 1\%$ is typical, as a validation size of $10,000$ samples is often statistically sufficient to evaluate model improvements.

In [ ]:
# ── 1. Train / Dev / Test Splits ──────────────────────────────────────

print("── Data Splitting Strategy ──")

X, y = make_classification(n_samples=10000, n_features=20, random_state=42)

# 60/20/20 split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42
)
X_dev, X_test, y_dev, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print(f"Train set: {X_train.shape[0]:>6} samples")
print(f"Dev set:   {X_dev.shape[0]:>6} samples")
print(f"Test set:  {X_test.shape[0]:>6} samples")

## 2. Evaluation Metrics

For binary classification, relying solely on **Accuracy** can be misleading, particularly for skewed class distributions. We calculate more specific metrics:

- **Precision (Positive Predictive Value)**: Of all positive predictions, what fraction was actually positive?
  $$\text{Precision} = \frac{TP}{TP + FP}$$
- **Recall (Sensitivity)**: Of all actual positive samples, what fraction did the model correctly find?
  $$\text{Recall} = \frac{TP}{TP + FN}$$
- **F1 Score**: The harmonic mean of precision and recall. It balances both metrics and provides a single value metric:
  $$\text{F1} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$
- **ROC Curve (Receiver Operating Characteristic)**: Plots the True Positive Rate (Recall) vs False Positive Rate ($\frac{FP}{TN+FP}$) at various classification thresholds. The **Area Under Curve (AUC)** measures the overall probability that the model ranks a random positive sample higher than a random negative one.

In [ ]:
# ── 2. Evaluation Metrics ─────────────────────────────────────────────

print("\n── Classification Metrics ──")

model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_dev)
y_prob = model.predict_proba(X_dev)[:, 1]

print(f"Accuracy:  {accuracy_score(y_dev, y_pred):.3f}")
print(f"Precision: {precision_score(y_dev, y_pred):.3f}")
print(f"Recall:    {recall_score(y_dev, y_pred):.3f}")
print(f"F1 Score:  {f1_score(y_dev, y_pred):.3f}")

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_dev, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["Negative", "Positive"]).plot()
plt.title("Confusion Matrix — Dev Set")
plt.show()

# ROC curve
fpr, tpr, _ = roc_curve(y_dev, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, "b-", linewidth=2, label=f"ROC (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 3. Cross-Validation

In $k$-fold cross-validation, the training set is split into $k$ subsets. The model is trained $k$ times, each time using $k-1$ subsets for training and $1$ subset as validation. This yields a more stable, robust measure of validation accuracy variance, especially on smaller datasets.

In [ ]:
# ── 3. Cross-Validation ───────────────────────────────────────────────

print("\n── Cross-Validation (5-fold) ──")
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")
print(f"CV scores:        {cv_scores.round(3)}")
print(f"Mean ± Std:       {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

## 4. Error Analysis

Error analysis is the manual inspection of samples that the model misclassified. By categorizing errors into specific themes (e.g. noise, blurry, low lighting), you can prioritize your development effort based on which category accounts for the highest percentage of incorrect predictions.

In [ ]:
# ── 4. Error Analysis Template ────────────────────────────────────────

print("\n── Error Analysis Checklist ──")
print("""
When your model makes errors on the dev set:

1. Group errors by category (e.g., blurry image, occluded, low lighting)
2. Count how many fall into each category
3. Prioritise the category with the most errors

Example table:
┌──────────────────────┬───────┬────────────┐
│ Error Category       │ Count │ % of Total │
├──────────────────────┼───────┼────────────┤
│ Blurry image         │    12 │        40% │ ← START HERE
│ Occluded object      │     8 │        27% │
│ Low lighting         │     7 │        23% │
│ Other                │     3 │        10% │
└──────────────────────┴───────┴────────────┘
""")

## 5. Class Imbalance

When class distributions are highly skewed (e.g. only $10\%$ positive), a classifier can achieve $90\%$ accuracy by simply predicting $0$ every time. To handle class imbalance, we evaluate using Precision, Recall, and F1 instead of Accuracy. We can also adjust the loss function using cost-sensitive weights (`class_weight='balanced'`), which penalizes minority class errors more heavily.

In [ ]:
# ── 5. Class Imbalance Demo ───────────────────────────────────────────

print("── Class Imbalance ──")

X_imb, y_imb = make_classification(
    n_samples=1000, weights=[0.9, 0.1], random_state=42
)
print(f"Class distribution: {np.bincount(y_imb)}  ({(y_imb == 1).mean():.1%} positive)")

X_imb_train, X_imb_test, y_imb_train, y_imb_test = train_test_split(
    X_imb, y_imb, test_size=0.3, random_state=42
)

imb_model = RandomForestClassifier(n_estimators=50, class_weight="balanced",
                                    random_state=42)
imb_model.fit(X_imb_train, y_imb_train)
y_imb_pred = imb_model.predict(X_imb_test)

print(f"Accuracy:  {accuracy_score(y_imb_test, y_imb_pred):.3f}")
print(f"Precision: {precision_score(y_imb_test, y_imb_pred):.3f}")
print(f"Recall:    {recall_score(y_imb_test, y_imb_pred):.3f}")
print(f"F1 Score:  {f1_score(y_imb_test, y_imb_pred):.3f}")
print("  → For imbalanced data, Accuracy is misleading!")
print("  → Use Precision / Recall / F1 instead.")

## Key Takeaways

- **Train / Dev / Test Allocation**: Use $60/20/20$ splits or $98/1/1$ (for big data). Keep dev and test sets in the same distribution.
- **Unbiased Evaluation**: The test set is only used once at the very end to estimate generalization error. Peeking at test set performance during tuning introduces model bias.
- **Bias/Variance Metrics**:
  - **Avoidable Bias** = Train Error - Bayes (Human-level) Error.
  - **Variance** = Dev Error - Train Error.
- **Skewed Data Strategy**: Use F1 score or Area Under the ROC Curve instead of Accuracy. Set `class_weight='balanced'` in models to handle class skew.
- **Error Analysis**: Manually categorize misclassifications on the dev set to identify which feature engineering tasks or data gathering efforts will yield the highest performance gains.

In [ ]:
# ── Key Takeaways ─────────────────────────────────────────────────────
print("""
╔══ Key Takeaways ───────────────────────────────────────────────╗
║ • Train/Dev/Test: 60/20/20 or 98/1/1 for big data            ║
║ • Dev set = iterate & tune hyperparams                       ║
║ • Test set = final evaluation ONLY (peeking leads to bias)    ║
║ • Human-level performance ≈ Bayes error (upper bound)        ║
║ • Avoidable bias  = train error - human error                ║
║ • Variance        = dev error - train error                  ║
║ • For imbalanced data: use Precision, Recall, F1, ROC-AUC    ║
║ • class_weight='balanced' auto-adjusts for imbalance         ║
╚════════════════════════════════════════════════════════════════╝
""")